In [ ]:
# 编码器： 把文本编码成数字的工具。 

In [9]:
sentence = 'hello everyone , today is a good day .'

In [2]:
# 字典， 也叫做词表 vocabulary
vocab = {
    '<SOS>': 0,
    '<EOS>': 1,
    'hello': 2,
    'everyone': 3,
    'today': 4,
    'is': 5,
    'a': 6,
    'good': 7,
    'day': 8,
    ',': 9,
    '.': 10
}

In [10]:
sent = '<SOS> ' + sentence + ' <EOS>'
print(sent)

<SOS> hello everyone , today is a good day . <EOS>


In [11]:
# 英文分词， 比较简单， 直接按照空格区分就可以。
# 中文可以使用分词工具，比如jieba分词。 
words = sent.split()
print(words)

['<SOS>', 'hello', 'everyone', ',', 'today', 'is', 'a', 'good', 'day', '.', '<EOS>']


In [12]:
[vocab[i] for i in words]

[0, 2, 3, 9, 4, 5, 6, 7, 8, 10, 1]

### 使用编码工具

In [13]:
# 模型和它的编码器是成对使用的， 你使用什么模型， 就它提供的编码器。
# 编码器的名字一般和模型的名字是一样的。 
# bert-base-chinese
from transformers import BertTokenizer

In [17]:
# 开了vpn之后。 如何在代码中使用vpn
import os

os.environ['http_proxy'] = '127.0.0.1:10809'
os.environ['https_proxy'] = '127.0.0.1:10809'

In [18]:
tokenizer = BertTokenizer.from_pretrained(
    pretrained_model_name_or_path='bert-base-chinese',
    cache_dir = None,
    force_download = False
)

D:\.venv\lib\site-packages\huggingface_hub\file_download.py:133: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SupercoldZzz\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [19]:
sents = [
    '你站在桥上看风景',
    '看风景的人在楼上看你',
    '明月装饰了你的窗子',
    '你装饰了别人的梦'
]

In [20]:
# 基本的编码函数
out = tokenizer.encode(
    text=sents[0],
    text_pair=sents[1],
    # 句子太长就截断到max_length
    truncation=True,
    # 句子不够长就padding到max_length的长度
    padding='max_length',
    add_special_tokens=True,
    max_length=25,
    return_tensors=None
)
print(out)

[101, 872, 4991, 1762, 3441, 677, 4692, 7599, 3250, 102, 4692, 7599, 3250, 4638, 782, 1762, 3517, 677, 4692, 872, 102, 0, 0, 0, 0]


In [21]:
# 把数字还原成字符串
tokenizer.decode(out)

'[CLS] 你 站 在 桥 上 看 风 景 [SEP] 看 风 景 的 人 在 楼 上 看 你 [SEP] [PAD] [PAD] [PAD] [PAD]'

In [22]:
# 进阶版编码函数
out = tokenizer.encode_plus(
    text=sents[0],
    text_pair=sents[1],
    truncation=True,
    padding='max_length',
    # 句子最大长度
    max_length=25,
    # 是否添加特殊字符, [cls]
    add_special_tokens=True,
    # 返回的数据类型, 默认返回列表, 可以返回Tensorflow, pytorch的tensor
    return_tensors=None,
    # 0, 1 表示是哪个句子的数据
    return_token_type_ids=True,
    # 有用部分标1, pad部分标0
    return_attention_mask=True,
    # 特殊字符标1, 其他位置标0
    return_special_tokens_mask=True,
    # 返回句子长度. 
    return_length=True
)

In [23]:
for k, v in out.items():
    print(k, ':', v)

input_ids : [101, 872, 4991, 1762, 3441, 677, 4692, 7599, 3250, 102, 4692, 7599, 3250, 4638, 782, 1762, 3517, 677, 4692, 872, 102, 0, 0, 0, 0]
token_type_ids : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0]
special_tokens_mask : [1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1]
attention_mask : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0]
length : 25


In [25]:
# 批量编码函数
out = tokenizer.batch_encode_plus(
    # 句子对
    # 如果要对单句子编码, batch_text_or_text_pairs=[sents[0], sents[1], sents[2], ...]
    batch_text_or_text_pairs=[(sents[0], sents[1]), (sents[2], sents[3])],
    truncation=True,
    padding='max_length',
    max_length=25,
    add_special_tokens=True,
    return_tensors=None,
    return_token_type_ids=True,
    return_attention_mask=True,
    return_special_tokens_mask=True,
    return_length=True
)

In [26]:
for k, v in out.items():
    print(k, ': ', v)

input_ids :  [[101, 872, 4991, 1762, 3441, 677, 4692, 7599, 3250, 102, 4692, 7599, 3250, 4638, 782, 1762, 3517, 677, 4692, 872, 102, 0, 0, 0, 0], [101, 3209, 3299, 6163, 7652, 749, 872, 4638, 4970, 2094, 102, 872, 6163, 7652, 749, 1166, 782, 4638, 3457, 102, 0, 0, 0, 0, 0]]
token_type_ids :  [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0]]
special_tokens_mask :  [[1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1], [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1]]
length :  [21, 20]
attention_mask :  [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0]]


In [27]:
# 字典的操作
# 获取字典
vocab = tokenizer.get_vocab()

In [28]:
vocab

{'[PAD]': 0,
 '[unused1]': 1,
 '[unused2]': 2,
 '[unused3]': 3,
 '[unused4]': 4,
 '[unused5]': 5,
 '[unused6]': 6,
 '[unused7]': 7,
 '[unused8]': 8,
 '[unused9]': 9,
 '[unused10]': 10,
 '[unused11]': 11,
 '[unused12]': 12,
 '[unused13]': 13,
 '[unused14]': 14,
 '[unused15]': 15,
 '[unused16]': 16,
 '[unused17]': 17,
 '[unused18]': 18,
 '[unused19]': 19,
 '[unused20]': 20,
 '[unused21]': 21,
 '[unused22]': 22,
 '[unused23]': 23,
 '[unused24]': 24,
 '[unused25]': 25,
 '[unused26]': 26,
 '[unused27]': 27,
 '[unused28]': 28,
 '[unused29]': 29,
 '[unused30]': 30,
 '[unused31]': 31,
 '[unused32]': 32,
 '[unused33]': 33,
 '[unused34]': 34,
 '[unused35]': 35,
 '[unused36]': 36,
 '[unused37]': 37,
 '[unused38]': 38,
 '[unused39]': 39,
 '[unused40]': 40,
 '[unused41]': 41,
 '[unused42]': 42,
 '[unused43]': 43,
 '[unused44]': 44,
 '[unused45]': 45,
 '[unused46]': 46,
 '[unused47]': 47,
 '[unused48]': 48,
 '[unused49]': 49,
 '[unused50]': 50,
 '[unused51]': 51,
 '[unused52]': 52,
 '[unused53]': 53

In [29]:
len(vocab)

21128

In [30]:
# bert_base_chinese是把每个中文字当成一个词. 
'明月' in vocab

False

In [31]:
# 添加新词
tokenizer.add_tokens(new_tokens=['明月', '装饰', '窗子'])

3

In [32]:
# 添加特殊字符
tokenizer.add_special_tokens({'eos_token': '[EOS]'})

1

In [35]:
for word in ['明月', '装饰', '窗子', '[EOS]']:
    print(tokenizer.get_vocab()[word])

21128
21129
21130
21131


In [36]:
# 用新词表去编码
out = tokenizer.encode(text='明月装饰了你的窗子[EOS]',
                text_pair=None,
                truncation=True,
                padding='max_length',
                add_special_tokens=True,
                max_length=10,
                return_tensors=None)
print(out)

[101, 21128, 21129, 749, 872, 4638, 21130, 21131, 102, 0]


In [37]:
tokenizer.decode(out)

'[CLS] 明月 装饰 了 你 的 窗子 [EOS] [SEP] [PAD]'

In [ ]:
# 总结: 编码器工作流程: 定义字典, 句子预处理, 分词, 编码